In [1]:
# cell 1 — imports and load data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from pathlib import Path
from scipy import stats
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# style
sns.set_theme(style="darkgrid")

# connection
DB_PATH = Path("../data/database/steam.db")
conn = sqlite3.connect(DB_PATH)

# load data
games = pd.read_sql("SELECT * FROM games", conn)
genre_perf = pd.read_csv("../data/processed/genre_performance.csv")
price_tiers = pd.read_csv("../data/processed/price_tiers.csv")

# filter to games with meaningful ratings
games_filtered = games[games['total_ratings'] > 10].copy()

print(f"Total games: {len(games)}")
print(f"Games with >10 ratings: {len(games_filtered)}")

Total games: 27075
Games with >10 ratings: 20024


In [2]:
# cell 2 — Mann-Whitney U: free vs paid reach
free_games = games_filtered[games_filtered['price'] == 0]['owner_midpoint']
paid_games = games_filtered[games_filtered['price'] > 0]['owner_midpoint']

u_stat, p_value = stats.mannwhitneyu(free_games, paid_games, alternative='two-sided')

print(f"Free games: {len(free_games)} games, median owners: {free_games.median():,.0f}")
print(f"Paid games: {len(paid_games)} games, median owners: {paid_games.median():,.0f}")
print(f"\nMann-Whitney U statistic: {u_stat:,.0f}")
print(f"P-value: {p_value:.6f}")
print(f"\nResult: {'Statistically significant' if p_value < 0.05 else 'Not significant'} (α=0.05)")

Free games: 2272 games, median owners: 35,000
Paid games: 17752 games, median owners: 10,000

Mann-Whitney U statistic: 25,877,510
P-value: 0.000000

Result: Statistically significant (α=0.05)


In [3]:
# cell 3 — Kruskal-Wallis: satisfaction across genres
genre_satisfaction = pd.read_sql("""
    SELECT g.genre, gm.positive_ratio
    FROM game_genres g
    JOIN games gm ON g.appid = gm.appid
    WHERE gm.total_ratings > 10
    AND g.genre NOT IN (
        'Early Access', 'Free to Play', 'Indie',
        'Gore', 'Violent', 'Nudity', 'Sexual Content',
        'Animation & Modeling', 'Design & Illustration',
        'Utilities', 'Audio Production', 'Video Production',
        'Web Publishing', 'Education', 'Software Training'
    )
    AND gm.positive_ratio IS NOT NULL
""", conn)

# create a list of arrays, one per genre
groups = [
    group['positive_ratio'].values 
    for name, group in genre_satisfaction.groupby('genre')
]

h_stat, p_value = stats.kruskal(*groups)

print("=== Kruskal-Wallis Test ===")
print(f"H statistic: {h_stat:.2f}")
print(f"P-value: {p_value:.6f}")
print(f"Result: {'Statistically significant' if p_value < 0.05 else 'Not significant'} (α=0.05)")

print("\n=== Median satisfaction per genre ===")
print(genre_satisfaction.groupby('genre')['positive_ratio']
      .median().sort_values(ascending=False).round(3))

=== Kruskal-Wallis Test ===
H statistic: 499.76
P-value: 0.000000
Result: Statistically significant (α=0.05)

=== Median satisfaction per genre ===
genre
Adventure                0.769
Action                   0.763
RPG                      0.763
Game Development         0.761
Casual                   0.759
Strategy                 0.734
Sports                   0.734
Simulation               0.714
Racing                   0.714
Photo Editing            0.681
Massively Multiplayer    0.641
Accounting               0.417
Name: positive_ratio, dtype: float64
